ML ASSIGNMENT 2 - MODEL TRAINING
MUTUAL FUND PREDICTION

Import libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

In [8]:
df = pd.read_csv("Mutual_fund Data.csv")

In [10]:
df.head()

,AMC,Fund Name,Morning star rating,Value Research rating,1 month return,NAV,1 Year return,3 Year Return,Minimum investment,Fund Manager,AUM,Category,Risk
0,mahindra manulife mutual fund,Mahindra Manulife Large & Mid Cap Reg-G,3,3,5.51%,28.32,37.79%,20.29%,Rs.500.0,Abhinav Khandelwal,2569.63 cr,Equity,High
1,mahindra manulife mutual fund,Mahindra Manulife Consumption Reg-G,0,3,7.25%,24.89,46.08%,21.93%,Rs.500.0,Abhinav Khandelwal,322.26 cr,Equity,High
2,mahindra manulife mutual fund,Mahindra Manulife Mid Cap Reg-G,4,4,5.94%,35.11,54.46%,27.48%,Rs.500.0,Abhinav Khandelwal,3292.76 cr,Equity,High
3,mahindra manulife mutual fund,Mahindra Manulife Small Cap Reg-G,0,0,8.37%,21.40,59.79%,0,Rs.500.0,Abhinav Khandelwal,5278.7 cr,Equity,Very High
4,mahindra manulife mutual fund,Mahindra Manulife Large Cap Reg-G,4,3,4.06%,23.69,32.07%,15.18%,Rs.500.0,Abhinav Khandelwal,577.72 cr,Equity,Very High


Data Cleaning and Pre-processing

In [7]:
percent_cols = ['1 month return', '1 Year return', '3 Year Return']

for col in percent_cols:
    df[col] = df[col].str.replace('%', '', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce')


df['Minimum investment'] = (
    df['Minimum investment']
    .str.replace('Rs.', '', regex=False)
    .astype(float)
)



In [9]:
invalid_aum = df[
    pd.to_numeric(
        df['AUM'].str.replace(' cr', '', regex=False),
        errors='coerce'
    ).isna()
]

invalid_aum[['AUM']]

,AUM
44,#0ME?
316,#0ME?
714,#0ME?
784,#0ME?
803,#0ME?
804,#0ME?
821,#0ME?
1195,#0ME?
1210,#0ME?
1357,#0ME?


In [11]:
df = df[
    pd.to_numeric(
        df['AUM'].str.replace(' cr', '', regex=False),
        errors='coerce'
    ).notna()
].copy()


df['AUM'] = (
    df['AUM']
    .str.replace(' cr', '', regex=False)
)

df['AUM'] = pd.to_numeric(df['AUM'], errors='coerce')

In [13]:
df.shape

(1383, 13)

In [15]:
df.drop(columns=['Fund Name', 'Fund Manager'], inplace=True)

In [17]:
df.isna().sum()

AMC                      0
Morning star rating      0
Value Research rating    0
1 month return           0
NAV                      0
1 Year return            0
3 Year Return            0
Minimum investment       0
AUM                      0
Category                 0
Risk                     0
dtype: int64

In [19]:
df.describe()

,Morning star rating,Value Research rating,1 month return,NAV,1 Year return,3 Year Return,Minimum investment,AUM
count,1383.000000,1383.000000,1383.000000,1383.000000,1383.000000,1383.000000,1.383000e+03,1383.000000
mean,1.678959,1.681128,3.344931,323.466522,22.112336,8.774440,3.689455e+04,4737.091764
std,1.793512,1.730458,2.698776,859.595027,18.789963,8.758257,8.914018e+05,9658.903104
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000
25%,0.000000,0.000000,0.700000,15.380000,7.320000,0.000000,1.000000e+02,223.890000
50%,1.000000,2.000000,3.170000,29.080000,14.910000,5.850000,5.000000e+02,1169.590000
75%,3.000000,3.000000,5.420000,104.140000,37.295000,14.690000,5.000000e+02,4439.020000
max,5.000000,5.000000,12.490000,6706.890000,86.350000,39.470000,3.000000e+07,95391.460000


In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1383 entries, 0 to 1392
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   AMC                    1383 non-null   object 
 1   Morning star rating    1383 non-null   int64  
 2   Value Research rating  1383 non-null   int64  
 3   1 month return         1383 non-null   float64
 4   NAV                    1383 non-null   float64
 5   1 Year return          1383 non-null   float64
 6   3 Year Return          1383 non-null   float64
 7   Minimum investment     1383 non-null   float64
 8   AUM                    1383 non-null   float64
 9   Category               1383 non-null   object 
 10  Risk                   1383 non-null   object 
dtypes: float64(6), int64(2), object(3)
memory usage: 129.7+ KB


In [23]:
df['AMC'].nunique(), df['Category'].nunique()

(34, 4)

In [25]:
le_amc = LabelEncoder()
le_cat = LabelEncoder()

df['AMC'] = le_amc.fit_transform(df['AMC'])
df['Category'] = le_cat.fit_transform(df['Category'])

In [26]:
df[['AMC', 'Category']].head()

,AMC,Category
0,18,1
1,18,1
2,18,1
3,18,1
4,18,1


In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1383 entries, 0 to 1392
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   AMC                    1383 non-null   int64  
 1   Morning star rating    1383 non-null   int64  
 2   Value Research rating  1383 non-null   int64  
 3   1 month return         1383 non-null   float64
 4   NAV                    1383 non-null   float64
 5   1 Year return          1383 non-null   float64
 6   3 Year Return          1383 non-null   float64
 7   Minimum investment     1383 non-null   float64
 8   AUM                    1383 non-null   float64
 9   Category               1383 non-null   int64  
 10  Risk                   1383 non-null   object 
dtypes: float64(6), int64(4), object(1)
memory usage: 129.7+ KB


In [28]:
df['return_spread'] = df['1 Year return'] - df['3 Year Return']

In [33]:
df['momentum_ratio'] = df['1 month return'] / (df['1 Year return'] + 1)

In [35]:
df['aum_min_inv_ratio'] = df['AUM'] / (df['Minimum investment'] + 1)

In [37]:
df['Risk'] = df['Risk'].str.strip().str.lower()

In [39]:
risk_encoder = LabelEncoder()
df['Risk_encoded'] = risk_encoder.fit_transform(df['Risk'])

In [41]:
dict(zip(risk_encoder.classes_, risk_encoder.transform(risk_encoder.classes_)))

{'high': 0,
 'low': 1,
 'low to moderate': 2,
 'low tomoderate': 3,
 'moderate': 4,
 'moderately high': 5,
 'very high': 6}

In [43]:
df['Risk'].value_counts()

Risk
very high          543
moderate           325
low to moderate    164
high               153
moderately high    102
low                 91
low tomoderate       5
Name: count, dtype: int64

In [45]:
df['Risk'] = (
    df['Risk']
    .str.strip()
    .str.lower()
    .str.replace(r'\s+', ' ', regex=True)
)

In [47]:
final_risk_mapping = {
    'low': 'Low',
    'low to moderate': 'Moderate',
    'low tomoderate': 'Moderate',
    'moderate': 'Moderate',
    'moderately high': 'High',
    'high': 'High',
    'very high': 'Very High'
}

df['Risk'] = df['Risk'].replace(final_risk_mapping)

In [49]:
df['Risk'].value_counts()

Risk
Very High    543
Moderate     494
High         255
Low           91
Name: count, dtype: int64

In [51]:
risk_encoder = LabelEncoder()
df['Risk_encoded'] = risk_encoder.fit_transform(df['Risk'])

dict(zip(risk_encoder.classes_, risk_encoder.transform(risk_encoder.classes_)))

{'High': 0, 'Low': 1, 'Moderate': 2, 'Very High': 3}

In [53]:
X = df.drop(columns=['Risk', 'Risk_encoded'])
y = df['Risk_encoded']

In [55]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [57]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [61]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "XGBoost": XGBClassifier(
        objective='multi:softprob',
        eval_metric='mlogloss',
        num_class=len(y.unique()),
        random_state=42
    )
}

In [63]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, matthews_corrcoef
)

results = []

In [65]:
for name, model in models.items():
    
    # Use scaled data where needed
    if name in ["Logistic Regression", "KNN", "Naive Bayes"]:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob, multi_class='ovr'),
        "Precision": precision_score(y_test, y_pred, average='macro'),
        "Recall": recall_score(y_test, y_pred, average='macro'),
        "F1 Score": f1_score(y_test, y_pred, average='macro'),
        "MCC": matthews_corrcoef(y_test, y_pred)
    })